# Nivel 0: Conexión a OpenRouter

In [ ]:
!pip install requests -q

import requests
import json

OPENROUTER_API_KEY = "sk-or-v1-my-key"  # Reemplazá con tu key de https://openrouter.ai/keys
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "openai/gpt-4o-mini"  # Podés cambiar el modelo, ver https://openrouter.ai/models

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

# Nivel 1: Llamada básica

In [ ]:
mensaje_usuario = "Explicame en una frase qué es la estimación ágil de software."

payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": mensaje_usuario}
    ]
}

response = requests.post(OPENROUTER_URL, headers=HEADERS, json=payload)
data = response.json()

if response.status_code == 200:
    respuesta_texto = data["choices"][0]["message"]["content"]
    print("Respuesta del modelo:\n")
    print(respuesta_texto)
else:
    print("Error:", data)

# Nivel 2: Llamada con system_prompt

In [7]:
system_prompt = "Sos un experto en estimación de software con 15 años de experiencia. Respondé de forma técnica, precisa y estructurada."

# Llamada SIN system prompt (nivel 1)
payload_sin_rol = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": mensaje_usuario}
    ]
}

# Llamada CON system prompt (nivel 2)
payload_con_rol = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": mensaje_usuario}
    ]
}

resp_sin_rol = requests.post(OPENROUTER_URL, headers=HEADERS, json=payload_sin_rol).json()
resp_con_rol = requests.post(OPENROUTER_URL, headers=HEADERS, json=payload_con_rol).json()

texto_sin_rol = resp_sin_rol["choices"][0]["message"]["content"]
texto_con_rol = resp_con_rol["choices"][0]["message"]["content"]

print("=== SIN system prompt ===\n")
print(texto_sin_rol)
print("\n\n=== CON system prompt (experto) ===\n")
print(texto_con_rol)

print("\n\n=== Comparación rápida ===")
print(f"Longitud sin rol: {len(texto_sin_rol)} caracteres")
print(f"Longitud con rol: {len(texto_con_rol)} caracteres")

Respuesta del modelo:

La estimación ágil de software es un enfoque que utiliza técnicas colaborativas y flexibles para prever el esfuerzo y el tiempo necesarios para completar tareas o proyectos, adaptándose a cambios y promoviendo la comunicación continua entre los miembros del equipo.


# Nivel 3: metadatos y costos estimados

In [8]:
# Precios de referencia (USD por millón de tokens) - ACTUALIZAR según el modelo usado
# Podés consultar los precios reales por modelo en https://openrouter.ai/models
PRECIOS_POR_MODELO = {
    "openai/gpt-4o-mini": {"input": 0.15, "output": 0.60},
    # agregá más modelos según necesites
}

def calcular_costo(modelo, tokens_input, tokens_output):
    precios = PRECIOS_POR_MODELO.get(modelo)
    if not precios:
        return None
    costo_input = (tokens_input / 1_000_000) * precios["input"]
    costo_output = (tokens_output / 1_000_000) * precios["output"]
    return costo_input + costo_output

payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": mensaje_usuario}
    ]
}

response = requests.post(OPENROUTER_URL, headers=HEADERS, json=payload)
data = response.json()

respuesta_texto = data["choices"][0]["message"]["content"]
usage = data.get("usage", {})
modelo_usado = data.get("model", MODEL)

tokens_entrada = usage.get("prompt_tokens", 0)
tokens_salida = usage.get("completion_tokens", 0)
tokens_totales = usage.get("total_tokens", 0)

print("Respuesta:\n", respuesta_texto)
print("\n--- Metadatos ---")
print(f"Modelo utilizado: {modelo_usado}")
print(f"Tokens de entrada: {tokens_entrada}")
print(f"Tokens de salida: {tokens_salida}")
print(f"Tokens totales: {tokens_totales}")

costo = calcular_costo(MODEL, tokens_entrada, tokens_salida)
if costo is not None:
    print(f"Costo estimado: ${costo:.6f} USD")
else:
    print("No hay precio cargado para este modelo — agregalo a PRECIOS_POR_MODELO.")

Respuesta:
 La estimación ágil de software es un enfoque flexible y colaborativo que utiliza técnicas como el poker de planificación y puntos de historia para prever el esfuerzo necesario en el desarrollo de funcionalidades, favoreciendo la adaptación continua y la entrega incremental.

--- Metadatos ---
Modelo utilizado: openai/gpt-4o-mini
Tokens de entrada: 54
Tokens de salida: 47
Tokens totales: 101
Costo estimado: $0.000036 USD
